# 17b — Multimodel Classification Susceptibility Analysis

This notebook extends the LDA-only classification-susceptibility analysis to all retained models:

- LDA
- Logistic Regression
- EEGNet

The subject-level EEG features are not recomputed here. Instead, this notebook reuses the raw/ERP/power features computed in Notebook 17:

`results/thesis_artifacts/classification_susceptibility/tables/subject_raw_eeg_features.csv`

The goal is to test whether the same subject-level signal properties explain classification performance across different models.

Main questions:

1. Do baseline noise and ERP separability predict LDA, Logistic Regression, and EEGNet performance similarly?
2. Are some features model-specific predictors of classification success?
3. Can we define a general subject-level classifiability score across models?

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe
from oddball_eeg.stats import safe_corr, strength_word, zscore_within_group
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


set_pandas_display_options()

# ------------------------------------------------------------
# Experiments
# ------------------------------------------------------------

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

MODEL_NAMES = ["lda", "logreg", "eegnet"]

MODEL_LABELS = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}

# ------------------------------------------------------------
# Input from Notebook 17
# ------------------------------------------------------------

NOTEBOOK17_TABLE_DIR = Path("results") / "thesis_artifacts" / "classification_susceptibility" / "tables"
RAW_FEATURE_PATH = NOTEBOOK17_TABLE_DIR / "subject_raw_eeg_features.csv"

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "multimodel_classification_susceptibility"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Input raw feature table:", RAW_FEATURE_PATH)
print("Output root:", OUT_ROOT)

In [ ]:
def normalize_model_name(model_value: str) -> str:
    """
    Normalize model names from saved result CSVs.
    """
    s = str(model_value).strip().lower()

    aliases = {
        "lda": "lda",
        "linear_discriminant_analysis": "lda",
        "linear discriminant analysis": "lda",

        "logreg": "logreg",
        "logistic_regression": "logreg",
        "logistic regression": "logreg",
        "lr": "logreg",

        "eegnet": "eegnet",
        "eeg_net": "eegnet",
    }

    return aliases.get(s, s)


def normalize_subject_for_cfg(subject_value, cfg) -> str:
    """
    Normalize subject IDs so values such as 1 match config subjects such as '001'.
    """
    s = str(subject_value).strip()

    if s.endswith(".0"):
        s = s[:-2]

    if not s.isdigit():
        return s

    configured = [str(x) for x in cfg.subjects.subjects]
    digit_lengths = [len(x) for x in configured if str(x).isdigit()]

    if len(digit_lengths) == 0:
        return s

    width = max(digit_lengths)
    return s.zfill(width)


def dataset_label_from_cfg(cfg) -> str:
    """
    Human-readable dataset/condition label.
    """
    ds = cfg.dataset.dataset
    preset = cfg.dataset.event_preset

    if ds == "ds006480":
        return f"{ds}_{preset}"

    return ds


def load_subject_level_auc_for_models(exp_name: str, model_names: list[str]) -> pd.DataFrame:
    """
    Load subject-level AUC for all requested models from subject_summary_across_seeds.csv.
    """
    cfg = get_experiment(exp_name)

    summary_path = cfg.table_dir("summaries") / "subject_summary_across_seeds.csv"

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Missing subject summary table for {exp_name}:\n{summary_path}"
        )

    df = pd.read_csv(summary_path)

    required = {"subject", "model", "auc_seed_mean"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{summary_path} is missing required columns: {sorted(missing)}"
        )

    df = df.copy()
    df["model"] = df["model"].map(normalize_model_name)
    df["subject"] = df["subject"].map(lambda x: normalize_subject_for_cfg(x, cfg))

    df = df[df["model"].isin(model_names)].copy()

    if df.empty:
        raise RuntimeError(f"No requested model rows found in {summary_path}")

    out = pd.DataFrame({
        "experiment_name": exp_name,
        "dataset": cfg.dataset.dataset,
        "task": cfg.dataset.task,
        "event_preset": cfg.dataset.event_preset,
        "dataset_label": dataset_label_from_cfg(cfg),
        "subject": df["subject"],
        "model": df["model"],
        "model_label": df["model"].map(MODEL_LABELS),
        "auc": df["auc_seed_mean"].astype(float),
        "auc_seed_std": df.get("auc_seed_std", np.nan),
        "pr_auc": df.get("pr_auc_seed_mean", np.nan),
        "bal_acc": df.get("bal_acc_seed_mean", np.nan),
        "acc": df.get("acc_seed_mean", np.nan),
        "f1": df.get("f1_seed_mean", np.nan),
        "summary_path": str(summary_path),
    })

    return out.reset_index(drop=True)

In [ ]:
if not RAW_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find raw feature table from Notebook 17:\n{RAW_FEATURE_PATH}\n\n"
        "Run Notebook 17 first."
    )

raw_feature_df = pd.read_csv(RAW_FEATURE_PATH)

required_feature_cols = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
}

missing = required_feature_cols - set(raw_feature_df.columns)

if missing:
    raise ValueError(
        f"Raw feature table is missing required columns: {sorted(missing)}"
    )

raw_feature_df["subject"] = raw_feature_df["subject"].astype(str)

save_dataframe(
    raw_feature_df,
    TABLE_DIR / "copied_subject_raw_eeg_features.csv",
)

print("Raw feature rows:", len(raw_feature_df))
display(round_numeric_columns(raw_feature_df.head(), 4))

In [ ]:
auc_frames = []

for exp_name in EXPERIMENT_NAMES:
    auc_frames.append(load_subject_level_auc_for_models(exp_name, MODEL_NAMES))

multimodel_auc_df = pd.concat(auc_frames, ignore_index=True)

save_dataframe(
    multimodel_auc_df,
    TABLE_DIR / "multimodel_subject_auc.csv",
)

print("AUC rows:", len(multimodel_auc_df))
display(round_numeric_columns(multimodel_auc_df.head(20), 4))

print("\nSubjects per dataset/model:")
display(
    multimodel_auc_df
    .groupby(["dataset_label", "model"])["subject"]
    .nunique()
    .unstack()
)

In [ ]:
multimodel_subject_df = multimodel_auc_df.merge(
    raw_feature_df,
    on=["experiment_name", "dataset", "task", "event_preset", "dataset_label", "subject"],
    how="inner",
)

save_dataframe(
    multimodel_subject_df,
    TABLE_DIR / "multimodel_subject_susceptibility_features.csv",
)

print("Merged rows:", len(multimodel_subject_df))
print("Expected rows:", len(multimodel_auc_df))

display(round_numeric_columns(multimodel_subject_df.head(), 4))

print("\nRows per dataset/model:")
display(
    multimodel_subject_df
    .groupby(["dataset_label", "model"])["subject"]
    .nunique()
    .unstack()
)

In [ ]:
ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "model",
    "model_label",
    "summary_path",
    "roi_channels",
    "roi_method",
}

TARGET_COLS = {
    "auc",
    "auc_seed_std",
    "pr_auc",
    "bal_acc",
    "acc",
    "f1",
}

def get_candidate_feature_columns(df: pd.DataFrame) -> list[str]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    candidates = [
        c for c in numeric_cols
        if c not in ID_COLS
           and c not in TARGET_COLS
    ]

    # Prefer log-power features over raw power features.
    candidates = [
        c for c in candidates
        if not (c.startswith("power_") and not c.startswith("log_"))
    ]

    return candidates


feature_cols = get_candidate_feature_columns(multimodel_subject_df)

print("Candidate features:", len(feature_cols))
for c in feature_cols:
    print(" -", c)

In [ ]:
within_rows = []

for model, model_df in multimodel_subject_df.groupby("model"):
    for dataset_label, g in model_df.groupby("dataset_label"):
        for feature in feature_cols:
            pearson_r, pearson_p, n_p = safe_corr(
                g["auc"],
                g[feature],
                method="pearson",
            )

            spearman_rho, spearman_p, n_s = safe_corr(
                g["auc"],
                g[feature],
                method="spearman",
            )

            within_rows.append({
                "model": model,
                "model_label": MODEL_LABELS.get(model, model),
                "dataset_label": dataset_label,
                "feature": feature,
                "n": n_p,
                "pearson_r": pearson_r,
                "pearson_p": pearson_p,
                "spearman_rho": spearman_rho,
                "spearman_p": spearman_p,
                "abs_pearson_r": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
                "abs_spearman_rho": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
            })

within_corr_df = pd.DataFrame(within_rows)

within_corr_df = within_corr_df.sort_values(
    ["model", "dataset_label", "abs_spearman_rho"],
    ascending=[True, True, False],
).reset_index(drop=True)

save_dataframe(
    within_corr_df,
    TABLE_DIR / "within_dataset_correlations_by_model.csv",
)

display(round_numeric_columns(within_corr_df.head(40), 4))

In [ ]:
z_df = multimodel_subject_df.copy()

# Z-score AUC within each dataset/model pair.
z_df["auc_z"] = (
    z_df
    .groupby(["model", "dataset_label"])["auc"]
    .transform(zscore_within_group)
)

for feature in feature_cols:
    z_df[f"{feature}_z"] = (
        z_df
        .groupby(["model", "dataset_label"])[feature]
        .transform(zscore_within_group)
    )

save_dataframe(
    z_df,
    TABLE_DIR / "multimodel_subject_susceptibility_features_zscored.csv",
)

grand_rows = []

for model, model_df in z_df.groupby("model"):
    for feature in feature_cols:
        feature_z = f"{feature}_z"

        pearson_r, pearson_p, n_p = safe_corr(
            model_df["auc_z"],
            model_df[feature_z],
            method="pearson",
        )

        spearman_rho, spearman_p, n_s = safe_corr(
            model_df["auc_z"],
            model_df[feature_z],
            method="spearman",
        )

        valid = model_df[
            np.isfinite(model_df["auc_z"])
            & np.isfinite(model_df[feature_z])
            ]

        grand_rows.append({
            "model": model,
            "model_label": MODEL_LABELS.get(model, model),
            "feature": feature,
            "n": n_p,
            "pearson_r_z": pearson_r,
            "pearson_p_z": pearson_p,
            "spearman_rho_z": spearman_rho,
            "spearman_p_z": spearman_p,
            "abs_pearson_r_z": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
            "abs_spearman_rho_z": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
            "n_valid_grand": int(len(valid)),
            "n_valid_datasets": int(valid["dataset_label"].nunique()),
            "valid_datasets": ",".join(sorted(valid["dataset_label"].unique())),
        })

grand_corr_df = pd.DataFrame(grand_rows)

grand_corr_df = grand_corr_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(
    grand_corr_df,
    TABLE_DIR / "zscored_grand_correlations_by_model.csv",
)

# Report-level filtering: enough valid subjects and at least 3 datasets.
grand_corr_report_df = grand_corr_df[
    (grand_corr_df["n_valid_grand"] >= 40)
    & (grand_corr_df["n_valid_datasets"] >= 3)
    ].copy()

grand_corr_report_df = grand_corr_report_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(
    grand_corr_report_df,
    TABLE_DIR / "zscored_grand_correlations_report_filtered_by_model.csv",
)

display(round_numeric_columns(grand_corr_report_df.groupby("model").head(15), 4))

In [ ]:
within_summary_df = (
    within_corr_df
    .groupby(["model", "feature"])
    .agg(
        mean_within_pearson_r=("pearson_r", "mean"),
        mean_abs_within_pearson_r=("abs_pearson_r", "mean"),
        mean_within_spearman_rho=("spearman_rho", "mean"),
        mean_abs_within_spearman_rho=("abs_spearman_rho", "mean"),
        n_datasets=("dataset_label", "nunique"),
    )
    .reset_index()
)

top_feature_df = grand_corr_report_df.merge(
    within_summary_df,
    on=["model", "feature"],
    how="left",
)

top_feature_df = top_feature_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(
    top_feature_df,
    TABLE_DIR / "top_susceptibility_features_by_model.csv",
)

display(round_numeric_columns(top_feature_df.groupby("model").head(15), 4))

In [ ]:
# Pivot model AUCs into columns.
auc_wide_df = multimodel_subject_df.pivot_table(
    index=["experiment_name", "dataset", "task", "event_preset", "dataset_label", "subject"],
    columns="model",
    values="auc",
    aggfunc="mean",
).reset_index()

auc_wide_df.columns.name = None

for m in MODEL_NAMES:
    if m not in auc_wide_df.columns:
        auc_wide_df[m] = np.nan

auc_wide_df["mean_auc_across_models"] = auc_wide_df[MODEL_NAMES].mean(axis=1)
auc_wide_df["std_auc_across_models"] = auc_wide_df[MODEL_NAMES].std(axis=1)
auc_wide_df["max_auc_across_models"] = auc_wide_df[MODEL_NAMES].max(axis=1)
auc_wide_df["min_auc_across_models"] = auc_wide_df[MODEL_NAMES].min(axis=1)

feature_once_df = raw_feature_df.copy()

general_df = auc_wide_df.merge(
    feature_once_df,
    on=["experiment_name", "dataset", "task", "event_preset", "dataset_label", "subject"],
    how="inner",
)

general_df["mean_auc_z"] = (
    general_df
    .groupby("dataset_label")["mean_auc_across_models"]
    .transform(zscore_within_group)
)

for feature in feature_cols:
    if feature in general_df.columns:
        general_df[f"{feature}_z"] = (
            general_df
            .groupby("dataset_label")[feature]
            .transform(zscore_within_group)
        )

general_rows = []

for feature in feature_cols:
    feature_z = f"{feature}_z"

    if feature_z not in general_df.columns:
        continue

    pearson_r, pearson_p, n_p = safe_corr(
        general_df["mean_auc_z"],
        general_df[feature_z],
        method="pearson",
    )

    spearman_rho, spearman_p, n_s = safe_corr(
        general_df["mean_auc_z"],
        general_df[feature_z],
        method="spearman",
    )

    valid = general_df[
        np.isfinite(general_df["mean_auc_z"])
        & np.isfinite(general_df[feature_z])
        ]

    general_rows.append({
        "target": "mean_auc_across_models",
        "feature": feature,
        "n": n_p,
        "n_valid_grand": int(len(valid)),
        "n_valid_datasets": int(valid["dataset_label"].nunique()),
        "valid_datasets": ",".join(sorted(valid["dataset_label"].unique())),
        "pearson_r_z": pearson_r,
        "pearson_p_z": pearson_p,
        "spearman_rho_z": spearman_rho,
        "spearman_p_z": spearman_p,
        "abs_spearman_rho_z": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
    })

general_corr_df = pd.DataFrame(general_rows)

general_corr_report_df = general_corr_df[
    (general_corr_df["n_valid_grand"] >= 40)
    & (general_corr_df["n_valid_datasets"] >= 3)
    ].copy()

general_corr_report_df = general_corr_report_df.sort_values(
    "abs_spearman_rho_z",
    ascending=False,
).reset_index(drop=True)

save_dataframe(
    general_df,
    TABLE_DIR / "general_classifiability_subject_features.csv",
)

save_dataframe(
    general_corr_report_df,
    TABLE_DIR / "general_classifiability_correlations.csv",
)

display(round_numeric_columns(general_corr_report_df.head(20), 4))

In [ ]:
top_n = 15

for model in MODEL_NAMES:
    model_df = grand_corr_report_df[
        grand_corr_report_df["model"] == model
        ].head(top_n).copy()

    if model_df.empty:
        print(f"No reportable rows for {model}")
        continue

    model_df = model_df.sort_values("spearman_rho_z", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(
        model_df["feature"],
        model_df["spearman_rho_z"],
    )

    ax.axvline(0.0, linewidth=1)
    ax.set_xlabel("Spearman correlation with z-scored AUC")
    ax.set_title(f"Grand z-scored susceptibility correlations — {MODEL_LABELS[model]}")
    ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"grand_zscored_correlation_barplot_{model}.png")
    plt.show()

In [ ]:
top_n = 20

for model in MODEL_NAMES:
    top_features = (
        grand_corr_report_df[grand_corr_report_df["model"] == model]
        .head(top_n)["feature"]
        .tolist()
    )

    if len(top_features) == 0:
        continue

    heat_df = within_corr_df[
        (within_corr_df["model"] == model)
        & (within_corr_df["feature"].isin(top_features))
        ].copy()

    heat_pivot = heat_df.pivot(
        index="feature",
        columns="dataset_label",
        values="spearman_rho",
    )

    heat_pivot = heat_pivot.loc[top_features]

    fig, ax = plt.subplots(figsize=(8, max(5, 0.35 * len(top_features))))

    im = ax.imshow(
        heat_pivot.values,
        aspect="auto",
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(np.arange(len(heat_pivot.columns)))
    ax.set_xticklabels(heat_pivot.columns, rotation=45, ha="right")

    ax.set_yticks(np.arange(len(heat_pivot.index)))
    ax.set_yticklabels(heat_pivot.index)

    ax.set_title(f"Within-dataset Spearman correlations — {MODEL_LABELS[model]}")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Feature")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Spearman rho")

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"within_dataset_correlation_heatmap_{model}.png")
    plt.show()

In [ ]:
for model in MODEL_NAMES:
    model_top_features = (
        grand_corr_report_df[grand_corr_report_df["model"] == model]
        .head(4)["feature"]
        .tolist()
    )

    model_z_df = z_df[z_df["model"] == model].copy()

    for feature in model_top_features:
        feature_z = f"{feature}_z"

        if feature_z not in model_z_df.columns:
            continue

        row = grand_corr_report_df[
            (grand_corr_report_df["model"] == model)
            & (grand_corr_report_df["feature"] == feature)
            ].iloc[0]

        fig, ax = plt.subplots(figsize=(7, 5))

        for dataset_label, g in model_z_df.groupby("dataset_label"):
            ax.scatter(
                g[feature_z],
                g["auc_z"],
                label=dataset_label,
                alpha=0.8,
            )

        ax.axhline(0.0, linewidth=1)
        ax.axvline(0.0, linewidth=1)

        ax.set_xlabel(f"{feature} z-scored within dataset")
        ax.set_ylabel(f"{MODEL_LABELS[model]} AUC z-scored within dataset")
        ax.set_title(
            f"{MODEL_LABELS[model]} susceptibility\n"
            f"{feature}: rho={row['spearman_rho_z']:.3f}, "
            f"p={row['spearman_p_z']:.3g}"
        )
        ax.grid(True, alpha=0.3)
        ax.legend()

        fig.tight_layout()
        save_figure(fig, FIG_DIR / f"grand_z_scatter_{model}_{feature}.png")
        plt.show()

In [ ]:
# Compare the top reportable features across models.
model_comparison_rows = []

for feature in sorted(grand_corr_report_df["feature"].unique()):
    row = {"feature": feature}

    for model in MODEL_NAMES:
        sub = grand_corr_report_df[
            (grand_corr_report_df["model"] == model)
            & (grand_corr_report_df["feature"] == feature)
            ]

        if len(sub) == 0:
            row[f"{model}_rho"] = np.nan
            row[f"{model}_p"] = np.nan
            row[f"{model}_rank"] = np.nan
        else:
            idx = sub.index[0]
            model_rank = (
                grand_corr_report_df[grand_corr_report_df["model"] == model]
                .reset_index(drop=True)
                .query("feature == @feature")
                .index
            )

            row[f"{model}_rho"] = float(sub.iloc[0]["spearman_rho_z"])
            row[f"{model}_p"] = float(sub.iloc[0]["spearman_p_z"])
            row[f"{model}_rank"] = int(model_rank[0] + 1) if len(model_rank) else np.nan

    available_rhos = [
        row.get(f"{model}_rho", np.nan)
        for model in MODEL_NAMES
    ]

    row["mean_abs_rho_across_models"] = float(np.nanmean(np.abs(available_rhos)))
    row["mean_rho_across_models"] = float(np.nanmean(available_rhos))

    model_comparison_rows.append(row)

model_comparison_df = pd.DataFrame(model_comparison_rows)

model_comparison_df = model_comparison_df.sort_values(
    "mean_abs_rho_across_models",
    ascending=False,
).reset_index(drop=True)

save_dataframe(
    model_comparison_df,
    TABLE_DIR / "model_comparison_top_features.csv",
)

display(round_numeric_columns(model_comparison_df.head(30), 4))

In [ ]:
print("=" * 90)
print("Multimodel susceptibility summary")
print("=" * 90)

for model in MODEL_NAMES:
    print(f"\n{MODEL_LABELS[model]} top grand z-scored features:")

    model_top = grand_corr_report_df[
        grand_corr_report_df["model"] == model
        ].head(10)

    for _, row in model_top.iterrows():
        rho = row["spearman_rho_z"]
        p = row["spearman_p_z"]
        feature = row["feature"]

        print(
            f" - {feature}: rho={rho:.3f}, p={p:.3g}, "
            f"{strength_word(abs(rho))}"
        )

print("\nGeneral classifiability top features:")
for _, row in general_corr_report_df.head(10).iterrows():
    print(
        f" - {row['feature']}: rho={row['spearman_rho_z']:.3f}, "
        f"p={row['spearman_p_z']:.3g}, "
        f"{strength_word(abs(row['spearman_rho_z']))}"
    )

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)